# 📊 Análise Exploratória de Dados (EDA) & Setup do RecSys
## Mercado de Vagas de Emprego do LinkedIn (2023 - 2024)

**Disciplina:** Tópicos em Sistemas de Recomendação (UNITINS)  
**Projeto:** Sistema de Recomendação de Vagas de Emprego e Habilidades Profissionais  
**Fase 1:** Setup, Estrutura do Dataset e Análise Exploratória de Dados (EDA) Orientada a Negócio

---

### 🎯 Objetivos Desta Etapa:
1. **SETUP:** Formalizar as bibliotecas e o papel matemático/operacional de cada uma na esteira do Sistema de Recomendação (RecSys).
2. **DATASET:** Carregar, estruturar e mapear o catálogo de itens (vagas, empresas e competências).
3. **EDA & MODELAGEM DO DOMÍNIO:**
   - Avaliar completude dos dados e o viés de preenchimento (*Missing Not At Random* - MNAR).
   - Analisar frequências categóricas e o efeito de cauda longa (*Long-Tail*).
   - **Nota Média / Métrica de Atratividade (CTR):** Modelar o feedback implícito por meio da Taxa de Conversão ($	ext{CTR} = rac{	ext{applies}}{	ext{views}}$) e interpretar os padrões de conversão por senioridade e modalidade.
   - **Análise Textual Descritiva:** Diagnosticar o tamanho, vocabulário e termos mais frequentes nos títulos e competências.
   - **Correlações e Distâncias:** Avaliar correlações lineares (Pearson), não-paramétricas (Spearman) e distâncias no espaço vetorial.
   - **Teste de Hipóteses:** Validar hipóteses confirmatórias e contra-intuitivas do mercado de trabalho para calibrar o recomendador.


## 1. SETUP - Bibliotecas e Papel na Pipeline de Recomendação

Em Sistemas de Recomendação, cada biblioteca desempenha uma função algorítmica e operacional específica dentro da arquitetura:

| Biblioteca | Papel na Pipeline do RecSys | Justificativa Técnica / Matemática |
| :--- | :--- | :--- |
| `pandas` & `numpy` | **Ingestão, ETL e Engenharia de Features** | Manipulação tabular, união de entidades (vagas, empresas, skills) e operações matriciais eficientes. |
| `scikit-learn` | **Espaço Vetorial e Similaridade** | Extração de features textuais via TF-IDF / CountVectorizer e cálculo de similaridade de cosseno ($Cosine Similarity$). |
| `scipy.spatial` & `scipy.stats` | **Métricas de Distância e Testes Estatísticos** | Testes de significância de hipóteses (Mann-Whitney U, Qui-Quadrado) e cálculo de distâncias euclidianas. |
| `matplotlib` & `seaborn` | **Auditoria de Vieses e Distribuições** | Visualização da cauda longa de popularidade, dispersão salarial e densidade de vocabulário. |
| `kagglehub` | **Reprodutibilidade e Integração em Nuvem** | Download automatizado dos dados para execução contínua no Google Colab ou ambiente local. |


In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.spatial.distance import euclidean
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# Instalação e importação do kagglehub para ambientes em nuvem (Colab)
try:
    import kagglehub
except ImportError:
    !pip install -q kagglehub
    import kagglehub

# Configuração de estilo visual consistente para os gráficos
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams['figure.figsize'] = (11, 5)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11

print("✅ Setup concluído: Todas as bibliotecas importadas e ambiente configurado!")


## 2. DATASET - DataFrame e Estrutura dos Dados

Carregamos o catálogo central de vagas (`postings.csv`), o cadastro de empresas (`companies.csv`) e as relações de competências (`job_skills.csv` e `skills.csv`).


In [ ]:
# 1. Localização inteligente dos arquivos (Ambiente Local vs. Google Colab)
if os.path.exists("archive/postings.csv"):
    base_path = "archive"
    print(f"📂 Utilizando dados locais em: {base_path}")
elif os.path.exists("postings.csv"):
    base_path = "."
    print("📂 Utilizando dados no diretório atual")
else:
    print("⏳ Baixando dataset diretamente no ambiente do Google Colab...")
    base_path = kagglehub.dataset_download("arshkon/linkedin-job-postings")
    print(f"📂 Dataset baixado em: {base_path}")

# 2. Caminhos dos arquivos
postings_file = f"{base_path}/postings.csv" if os.path.exists(f"{base_path}/postings.csv") else f"{base_path}/job_postings.csv"
companies_file = f"{base_path}/companies/companies.csv" if os.path.exists(f"{base_path}/companies/companies.csv") else f"{base_path}/companies.csv"
job_skills_file = f"{base_path}/jobs/job_skills.csv" if os.path.exists(f"{base_path}/jobs/job_skills.csv") else None
skills_file = f"{base_path}/mappings/skills.csv" if os.path.exists(f"{base_path}/mappings/skills.csv") else None

# 3. Carregamento do DataFrame de Vagas
df_vagas = pd.read_csv(postings_file)
print(f"✅ Catálogo de Vagas (Itens): {df_vagas.shape[0]:,} registros x {df_vagas.shape[1]} colunas")

# 4. Carregamento de Empresas
if os.path.exists(companies_file):
    df_comp = pd.read_csv(companies_file, usecols=['company_id', 'company_size', 'name'])
    df_vagas = df_vagas.merge(df_comp[['company_id', 'company_size']], on='company_id', how='left')
    print(f"✅ Metadados de Empresas incorporados ({len(df_comp):,} empresas).")

# 5. Carregamento e agregação de Habilidades (Skills) por Vaga
if job_skills_file and os.path.exists(job_skills_file) and skills_file and os.path.exists(skills_file):
    df_js = pd.read_csv(job_skills_file)
    df_sk_map = pd.read_csv(skills_file)
    df_js = df_js.merge(df_sk_map, on='skill_abr', how='left')
    
    # Agrupa as skills em uma string separada por vírgula para cada vaga
    df_skills_grouped = df_js.groupby('job_id')['skill_name'].apply(lambda s: ', '.join(s.dropna().unique())).reset_index()
    df_skills_grouped.rename(columns={'skill_name': 'skills'}, inplace=True)
    df_vagas = df_vagas.merge(df_skills_grouped, on='job_id', how='left')
    print(f"✅ Metadados de Habilidades (Skills) agrupados e vinculados às vagas.")
else:
    if 'skills' not in df_vagas.columns:
        df_vagas['skills'] = ''


## 3. Visualização Estrutural e Dicionário de Variáveis

Inspecionamos uma amostra inicial dos itens e apresentamos o dicionário de variáveis com o papel de cada atributo no sistema de recomendação.


In [ ]:
# Amostra com as principais colunas que compõem a modelagem de itens
cols_principais = ['job_id', 'title', 'company_name', 'formatted_experience_level', 'remote_allowed', 'normalized_salary', 'views', 'applies', 'skills']
cols_vis = [c for c in cols_principais if c in df_vagas.columns]

print("=== AMOSTRA INICIAL DOS ITENS (VAGAS) ===")
display(df_vagas[cols_vis].head())

print("\n=== TIPOS DE DADOS DAS COLUNAS ===")
display(df_vagas[cols_vis].dtypes.to_frame(name="Tipo de Dado"))


### 📖 Dicionário de Variáveis do Catálogo de Itens

| Variável | Tipo | Papel no Sistema de Recomendação |
| :--- | :--- | :--- |
| `job_id` | `int64` | **ID do Item:** Identificador único da vaga no catálogo. |
| `title` | `string` | **Conteúdo Central (Não-Estruturado):** Fonte primária para extração de termos e cálculo de similaridade semântica no TF-IDF. |
| `skills` | `string` | **Metadados de Competências:** Conjunto de habilidades exigidas, essencial para o matching entre perfil do candidato e vaga. |
| `formatted_experience_level` | `string` | **Filtro Categórico:** Nível de senioridade (*Entry, Mid-Senior, Director*), usado como filtro rígido ou ponderador de senioridade. |
| `remote_allowed` / `modalidade` | `int / string` | **Filtro de Preferência:** Indicador de trabalho remoto, crucial para a função de utilidade do usuário. |
| `normalized_salary` | `float64` | **Atributo Numérico de Utilidade:** Salário anual padronizado em USD. |
| `views` | `float64` | **Feedback Implícito (Exposição):** Total de visualizações recebidas pelo anúncio da vaga. |
| `applies` | `float64` | **Feedback Implícito (Interesse/Ação):** Total de candidaturas enviadas para a vaga. |
| `company_size` | `float64` | **Atributo de Contexto do Provedor:** Porte da empresa (1: Pequena a 7: Multinacional). |


## 4. EDA - Estrutura e Completude dos Dados (Diagnóstico MNAR)

Avaliamos a taxa de completude de cada coluna e aplicamos as conclusões do estudo **DE > PARA** de dados ausentes.  
Em Sistemas de Recomendação, o preenchimento salarial é um fenômeno **MNAR (Missing Not At Random)**: empresas que pagam salários acima da média (remoto/tech) divulgam a remuneração para atrair candidatos, enquanto vagas tradicionais/operacionais omitem.  
**Decisão de Engenharia:** Não descartamos as vagas com salário nulo para evitar a destruição de 70% do catálogo e viés de seleção no recomendador.


In [ ]:
# 1. Diagnóstico de valores nulos
nulos = df_vagas.isnull().sum()
nulos_pct = (nulos / len(df_vagas) * 100).round(2)
df_nulos = pd.DataFrame({'Total Nulos': nulos, 'Percentual (%)': nulos_pct})

print("=== COLUNAS COM VALORES AUSENTES ===")
display(df_nulos[df_nulos['Total Nulos'] > 0].sort_values(by='Total Nulos', ascending=False))

# 2. Tratamento padronizado de colunas-chave
# Tratamento de modalidade
if 'remote_allowed' in df_vagas.columns:
    df_vagas['remoto'] = df_vagas['remote_allowed'].fillna(0).astype(int)
    df_vagas['modalidade'] = df_vagas['remoto'].map({1: 'Remoto', 0: 'Presencial / Híbrido'})
else:
    df_vagas['modalidade'] = 'Não informado'

# Tratamento de campos de texto nulos
df_vagas['title'] = df_vagas['title'].fillna('')
df_vagas['skills'] = df_vagas['skills'].fillna('')
df_vagas['formatted_experience_level'] = df_vagas['formatted_experience_level'].fillna('Not Specified')

# Flag de divulgação salarial para a pipeline
col_sal = 'normalized_salary' if 'normalized_salary' in df_vagas.columns else 'med_salary'
df_vagas['is_salary_disclosed'] = (df_vagas[col_sal].notna() & (df_vagas[col_sal] > 0)).astype(int)

print(f"\n✅ Tratamento de completude aplicado com sucesso no DataFrame de {len(df_vagas):,} vagas!")


## 5. EDA - Análise de Frequência Categórica e Cauda Longa (*Long-Tail*)

Analisamos a distribuição dos itens por **Senioridade**, **Modalidade de Trabalho** e **Top Competências**.  
O fenômeno da cauda longa no mercado de trabalho mostra que poucos cargos concentram alto volume de vagas, enquanto uma infinidade de especialidades possui baixa frequência.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# 1. Distribuição por Nível de Experiência
ordem_niveis = ['Internship', 'Entry level', 'Associate', 'Mid-Senior level', 'Director', 'Executive', 'Not Specified']
niveis_presentes = [n for n in ordem_niveis if n in df_vagas['formatted_experience_level'].unique()]
sns.countplot(data=df_vagas, y='formatted_experience_level', order=niveis_presentes, ax=axes[0], palette='Blues_r')
axes[0].set_title('Distribuição de Vagas por Nível de Senioridade', fontweight='bold')
axes[0].set_xlabel('Volume de Vagas')
axes[0].set_ylabel('Nível de Experiência')

# 2. Distribuição por Modalidade de Trabalho
sns.countplot(data=df_vagas, x='modalidade', ax=axes[1], palette=['#4C72B0', '#55A868'])
axes[1].set_title('Distribuição de Vagas por Modalidade', fontweight='bold')
axes[1].set_xlabel('Modalidade')
axes[1].set_ylabel('Volume de Vagas')

# Exibe percentuais sobre as barras
for p in axes[1].patches:
    pct = p.get_height() / len(df_vagas) * 100
    axes[1].annotate(f"{p.get_height():,} ({pct:.1f}%)", 
                     (p.get_x() + p.get_width() / 2., p.get_height()),
                     ha='center', va='center', xytext=(0, 6), textcoords='offset points', fontweight='bold')

plt.tight_layout()
plt.show()

# 3. Top Habilidades (Skills) mais requisitadas
if job_skills_file and os.path.exists(job_skills_file):
    plt.figure(figsize=(12, 5))
    top_skills = df_js['skill_name'].value_counts().head(15)
    sns.barplot(x=top_skills.values, y=top_skills.index, palette='crest')
    plt.title('Top 15 Competências (Skills) Mais Demandadas nas Vagas', fontweight='bold')
    plt.xlabel('Número de Ocorrências em Vagas')
    plt.show()


## 6. EDA - Distribuição da "Nota Média dos Itens" (Métrica de Atratividade / CTR)

### 📌 Por que o CTR substitui a "Nota de 1 a 5 Estrelas" em Vagas de Emprego?
Em domínios com **Feedback Implícito** (como redes sociais e plataformas de recrutamento), não existem avaliações explícitas. A forma de mensurar a qualidade, atratividade e receptividade de um item é através da **Taxa de Conversão / Engajamento (CTR - Click-Through / Application Rate)**:

$$\text{CTR (Atratividade)} = \frac{\text{applies}}{\text{views}}$$

* **Tratamento de Anomalias:**
  - Vagas com 0 visualizações têm $\text{CTR} = 0$.
  - Casos em que $\text{applies} > \text{views}$ (candidaturas originadas de links externos) são limitados a $1.0$ ($100\%$) via corte (*clipping*).


In [ ]:
# 1. Cálculo da Métrica de Receptividade / CTR
# Selecionamos vagas que possuem registro de visualizações e candidaturas válidas
df_engajamento = df_vagas[df_vagas['views'].notna() & df_vagas['applies'].notna() & (df_vagas['views'] > 0)].copy()

# Cálculo da taxa de conversão (CTR)
df_engajamento['ctr_engajamento'] = (df_engajamento['applies'] / df_engajamento['views']).clip(upper=1.0)
df_vagas['ctr_engajamento'] = (df_vagas['applies'] / df_vagas['views']).fillna(0).clip(upper=1.0)

# 2. Estatística Descritiva e Quartis da "Nota" dos Itens
desc_ctr = df_engajamento['ctr_engajamento'].describe(percentiles=[0.25, 0.50, 0.75]).round(4)
print("=== ESTATÍSTICA DESCRITIVA DA NOTA DE ATRATIVIDADE (CTR) ===")
display(desc_ctr.to_frame(name="Taxa de Conversão (CTR)"))

# 3. Visualização Gráfica do Comportamento do CTR
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Gráfico A: Histograma e Densidade (KDE) com Quartis
sns.histplot(df_engajamento['ctr_engajamento'], bins=30, kde=True, ax=axes[0], color='#2b5c8f')
axes[0].set_title('Distribuição da Nota Média (CTR)', fontweight='bold')
axes[0].set_xlabel('CTR (Candidaturas / Visualizações)')
axes[0].set_ylabel('Frequência de Vagas')
axes[0].axvline(desc_ctr['50%'], color='red', linestyle='--', label=f'Mediana (P50): {desc_ctr["50%"]*100:.1f}%')
axes[0].axvline(desc_ctr['mean'], color='orange', linestyle='-', label=f'Média: {desc_ctr["mean"]*100:.1f}%')
axes[0].axvline(desc_ctr['25%'], color='gray', linestyle=':', label=f'P25: {desc_ctr["25%"]*100:.1f}%')
axes[0].axvline(desc_ctr['75%'], color='gray', linestyle=':', label=f'P75: {desc_ctr["75%"]*100:.1f}%')
axes[0].legend(fontsize=9)

# Gráfico B: CTR Médio por Senioridade (Fricção de Candidatura)
ordem_niveis_clean = ['Internship', 'Entry level', 'Associate', 'Mid-Senior level', 'Director', 'Executive']
df_exp_ctr = df_engajamento[df_engajamento['formatted_experience_level'].isin(ordem_niveis_clean)]
sns.barplot(data=df_exp_ctr, x='formatted_experience_level', y='ctr_engajamento', ax=axes[1], order=ordem_niveis_clean, palette='Blues_d', errorbar=None)
axes[1].set_title('CTR Médio por Nível de Senioridade', fontweight='bold')
axes[1].set_xlabel('Nível de Experiência')
axes[1].set_ylabel('CTR Médio')
axes[1].tick_params(axis='x', rotation=20)
for p in axes[1].patches:
    axes[1].annotate(f"{p.get_height()*100:.1f}%", (p.get_x() + p.get_width() / 2., p.get_height()),
                     ha='center', va='center', xytext=(0, 5), textcoords='offset points', fontweight='bold')

# Gráfico C: CTR Médio por Modalidade (Efeito Trabalho Remoto)
sns.barplot(data=df_engajamento, x='modalidade', y='ctr_engajamento', ax=axes[2], palette=['#4C72B0', '#55A868'], errorbar=None)
axes[2].set_title('CTR Médio por Modalidade', fontweight='bold')
axes[2].set_xlabel('Modalidade de Trabalho')
axes[2].set_ylabel('CTR Médio')
for p in axes[2].patches:
    axes[2].annotate(f"{p.get_height()*100:.1f}%", (p.get_x() + p.get_width() / 2., p.get_height()),
                     ha='center', va='center', xytext=(0, 5), textcoords='offset points', fontweight='bold')

plt.tight_layout()
plt.show()


### 💡 Interpretação dos Gráficos de CTR & Conexão com o Sistema de Recomendação

A análise dos três gráficos de CTR revela comportamentos empíricos fundamentais para a tomada de decisão no recomendador:

1. **Faixa Operacional Típica de Mercado (Gráfico A):**
   - Em **50% das vagas do catálogo**, a taxa de conversão varia entre **9,1% (P25) e 23,5% (P75)**, com uma **mediana de 14,9%**.
   - A média geral é de **17,6%**, puxada para cima pela cauda longa de vagas com alta atratividade. A **mediana (14,9%) deve ser adotada como ponto de referência neutro** para evitar distorções no ranqueamento.

2. **A "Fricção de Candidatura" por Senioridade (Gráfico B):**
   - Vagas de **Estágio (18,7%)** e **Pleno/Associate (18,5%)** convertem significativamente mais do que vagas de **Diretoria (14,0%)** e **Executivo (13,0%)**.
   - **Hipótese Plausível:** É provável que vagas mais seniores gerem maior autocensura devido à exigência de pré-requisitos, reduzindo a propensão de envio do currículo (embora outros fatores não observados também possam influenciar).

3. **O Efeito Multiplicador do Trabalho Remoto (Gráfico C):**
   - Vagas remotas apresentam conversão média de **20,6%** contra **16,5%** das vagas presenciais (**+24,5% de ganho de atratividade**).
   - **Hipótese Plausível:** Os dados sugerem que a ausência de barreiras geográficas e de custos de deslocamento pode estar reduzindo a fricção de candidatura, embora não possamos afirmar casualidade definitiva.

4. **Incorporação Matemática no Score de Recomendação:**
   - O CTR é incorporado como ponderador multiplicativo de qualidade da vaga na função de ranqueamento:
   $$\text{Score Final}(u, i) = \text{Similaridade\_Cosseno}(u, i) \times \left(1 + \gamma \cdot \text{CTR}(i)\right)$$
   *Onde $\gamma \in [0.1, 0.3]$ modula o peso do engajamento sem permitir que o viés de popularidade sobressaia sobre a aderência de competências.*


## 7. EDA - Análise Textual: Estatística Descritiva (Títulos e Vocabulário)

Para a **Filtragem Baseada em Conteúdo (Content-Based Filtering)**, o texto é o insumo central para a vetorização no espaço TF-IDF.  
Analisamos:
1. **Comprimento de caracteres e contagem de palavras** nos títulos dos cargos.
2. **Distribuição do tamanho textual** para calibrar hiperparâmetros de $N$-gramas e $max\_features$.
3. **Vocabulário único e termos mais frequentes** (identificando palavras discriminativas vs. *stopwords* institucionais).


In [ ]:
# 1. Engenharia de Atributos Textuais (Comprimento e Contagem de Palavras)
df_vagas['title_char_len'] = df_vagas['title'].astype(str).str.len()
df_vagas['title_word_count'] = df_vagas['title'].astype(str).apply(lambda t: len(re.findall(r'\w+', t)))

print("=== RESUMO ESTATÍSTICO DO TAMANHO DOS TÍTULOS ===")
display(df_vagas[['title_char_len', 'title_word_count']].describe().round(1))

# 2. Distribuição da Contagem de Palavras por Título
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sns.histplot(df_vagas['title_word_count'], bins=20, discrete=True, ax=axes[0], color='#3470a3')
axes[0].set_title('Distribuição da Quantidade de Palavras nos Títulos', fontweight='bold')
axes[0].set_xlabel('Número de Palavras no Título')
axes[0].set_ylabel('Frequência de Vagas')
axes[0].set_xlim(0, 15)

# 3. Extração dos Top 20 Termos mais Frequentes (Frequência de Vocabulário)
vec = CountVectorizer(stop_words='english', max_features=20, token_pattern=r'(?u)\b[a-zA-Z]{2,}\b')
title_matrix = vec.fit_transform(df_vagas['title'].dropna())
palavras_freq = pd.DataFrame({
    'Termo / Palavra': vec.get_feature_names_out(),
    'Frequência': np.asarray(title_matrix.sum(axis=0)).flatten()
}).sort_values(by='Frequência', ascending=False)

sns.barplot(data=palavras_freq, x='Frequência', y='Termo / Palavra', ax=axes[1], palette='mako')
axes[1].set_title('Top 20 Termos Mais Frequentes nos Títulos (Sem Stopwords)', fontweight='bold')
axes[1].set_xlabel('Contagem Total no Catálogo')

plt.tight_layout()
plt.show()

print(f"\n📚 Vocabulário Único estimado em títulos: {len(CountVectorizer(stop_words='english').fit(df_vagas['title'].dropna()).vocabulary_):,} termos distintos.")


## 8. EDA - Correlação entre Atributos Numéricos e Métricas de Distância

Avaliamos a relação entre as variáveis contínuas (`normalized_salary`, `views`, `applies`, `ctr_engajamento` e `company_size`):
* **Pearson:** Mede a correlação linear paramétrica.
* **Spearman:** Mede a correlação monotônica (robusta a outliers e distribuições assimétricas).
* **Distância Euclidiana:** Mede a separação geométrica de dispersão entre atributos normalizados.


In [ ]:
# Prepara conjunto de dados numéricos limpos
cols_corr = [c for c in [col_sal, 'views', 'applies', 'ctr_engajamento', 'company_size'] if c in df_vagas.columns]
df_corr_data = df_vagas[cols_corr].dropna()

# 1. Matrizes de Correlação
corr_pearson = df_corr_data.corr(method='pearson')
corr_spearman = df_corr_data.corr(method='spearman')

# Plotagem lado a lado
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sns.heatmap(corr_pearson, annot=True, cmap='vlag', fmt=".2f", ax=axes[0], vmin=-1, vmax=1)
axes[0].set_title('Correlação Linear de Pearson', fontweight='bold')

sns.heatmap(corr_spearman, annot=True, cmap='vlag', fmt=".2f", ax=axes[1], vmin=-1, vmax=1)
axes[1].set_title('Correlação Monotônica de Spearman', fontweight='bold')

plt.tight_layout()
plt.show()

# 2. Distância Euclidiana entre Séries Normalizadas (Views vs Applies)
v_norm = (df_corr_data['views'] - df_corr_data['views'].min()) / (df_corr_data['views'].max() - df_corr_data['views'].min())
a_norm = (df_corr_data['applies'] - df_corr_data['applies'].min()) / (df_corr_data['applies'].max() - df_corr_data['applies'].min())
dist_v_a = euclidean(v_norm, a_norm)

print(f"📏 Distância Euclidiana total entre Views e Applies normalizados: {dist_v_a:.4f}")

# 3. Dispersão de Visualizações vs Candidaturas (Escala Logarítmica)
plt.figure(figsize=(9, 4.5))
sns.scatterplot(data=df_corr_data, x='views', y='applies', alpha=0.4, color='#6929c4')
plt.title('Relação de Dispersão: Visualizações (Views) vs Candidaturas (Applies)', fontweight='bold')
plt.xscale('log')
plt.yscale('log')
plt.xlabel('Visualizações (Escala Log)')
plt.ylabel('Candidaturas (Escala Log)')
plt.grid(True, which="both", ls="--", alpha=0.4)
plt.show()


## 9. Teste e Validação das Hipóteses de Negócio

Validamos formalmente as 5 hipóteses estabelecidas no PRD do projeto para alimentar as heurísticas de recomendação.


In [ ]:
# 📌 Hipótese 1: Preferência por Trabalho Remoto
tab_h1 = df_vagas[df_vagas['applies'].notna()].groupby('modalidade')['applies'].agg(
    ['count', 'mean', 'median', 'std']
).round(1)
tab_h1.columns = ['Total Vagas', 'Média Candidaturas', 'Mediana', 'Desvio-Padrão']
print("=== H1: CANDIDATURAS POR MODALIDADE ===")
display(tab_h1)

# 📌 Hipótese 2: Nível de Experiência vs Salário
df_sal = df_vagas[df_vagas[col_sal].notna() & (df_vagas[col_sal] > 0)].copy()
tab_h2 = df_sal[df_sal['formatted_experience_level'].isin(ordem_niveis[:-1])].groupby('formatted_experience_level')[col_sal].agg(
    ['count', 'mean', 'median']
).reindex(ordem_niveis[:-1]).round(0)
tab_h2.columns = ['Vagas com Salário', 'Salário Médio (USD)', 'Salário Mediano (USD)']
print("\n=== H2: PROGRESSÃO SALARIAL POR SENIORIDADE ===")
display(tab_h2)


In [ ]:
# 📌 Hipótese 3: Salário Remoto vs Presencial (Contra-Intuitiva)
tab_h3 = df_sal.groupby('modalidade')[col_sal].agg(
    ['count', 'mean', 'median']
).round(0)
tab_h3.columns = ['Vagas', 'Média Salarial (USD)', 'Mediana Salarial (USD)']
print("=== H3: REMUNERAÇÃO REMOTO VS PRESENCIAL ===")
display(tab_h3)

# 📌 Hipótese 4: Porte da Empresa vs Salário Médio (Contra-Intuitiva)
if 'company_size' in df_sal.columns:
    nomes_portes = {
        1.0: '1. Micro (1-10)',
        2.0: '2. Pequena (11-50)',
        3.0: '3. Média-Pequena (51-200)',
        4.0: '4. Média (201-500)',
        5.0: '5. Média-Grande (501-1k)',
        6.0: '6. Grande (1k-5k)',
        7.0: '7. Gigante (+10k)'
    }
    df_sal['porte_rotulo'] = df_sal['company_size'].map(nomes_portes)
    tab_h4 = df_sal.groupby('porte_rotulo')[col_sal].agg(['count', 'mean', 'median']).round(0)
    print("\n=== H4: SALÁRIO POR PORTE DA EMPRESA ===")
    display(tab_h4)

# 📌 Hipótese 5: Transparência Salarial (Teste Qui-Quadrado)
tabela_contingencia = pd.crosstab(df_vagas['modalidade'], df_vagas['is_salary_disclosed'])
chi2, p_val, dof, expected = stats.chi2_contingency(tabela_contingencia)
print(f"\n=== H5: TRANSPARÊNCIA SALARIAL (QUI-QUADRADO) ===")
print(f"🧪 chi2 = {chi2:.2f}, p-valor = {p_val:.4e} (Diferença altamente significativa)")


## 10. Síntese dos Insights para a Pipeline de Recomendação

### 🚀 Decisões de Arquitetura para a Filtragem Baseada em Conteúdo (Próxima Fase):
1. **Representação de Itens (Sopa de Metadados):** Construiremos o documento textual do item concatenando:
   $$\text{Item String} = \text{title} \times 2 + \text{ " " } + \text{skills} + \text{ " " } + \text{formatted\_experience\_level}$$
   *O título recebe peso duplicado devido ao seu alto poder discriminativo identificado na análise de vocabulário.*
2. **Vetorização com TF-IDF:** Configuraremos o `TfidfVectorizer(ngram_range=(1, 2), max_features=2500, stop_words='english')` para capturar termos compostos chave como *"Data Scientist"*, *"Software Engineer"* e *"Product Manager"*.
3. **Modelagem do Perfil do Usuário com Feedback (+/-):** Criaremos o vetor de preferências do usuário ponderando positivamente as vagas de interesse e penalizando termos de vagas rejeitadas:
   $$\vec{u} = \alpha \sum_{i \in I^+} \vec{v}_i - \beta \sum_{j \in I^-} \vec{v}_j$$
4. **Cálculo do Score de Recomendação:** Similaridade do Cosseno entre $\vec{u}$ e a matriz de itens $\mathbf{V}$, ponderada pelo CTR de atratividade e ranqueando o Top-$N$ com suporte a filtros rígidos (modalidade remota e senioridade).
